# Create a PyTorch3D wheel with `cvenv`

PyTorch3D has no reliable prebuilt wheels, and a wheel is only valid where the
**python version, torch version, and CUDA build all match** the machine it was
compiled on. Colab's stack drifts over time, so a wheel that worked last month
fails with an `undefined symbol` error today.

This notebook **builds a PyTorch3D wheel against the current runtime, saves it to
Google Drive, and (optionally) installs it** — so you only ever pay the ~15–40 min
compile once. Reuse the saved wheel in later sessions in seconds, or hand it to
students. Run on a **GPU runtime** (Runtime → Change runtime type → GPU).

> ⚠️ Keep this tab active during the build — Colab drops idle sessions, and the
> wheel is only written at the end.

## 1. Install `cvenv`

In [ ]:
!pip install -q "git+https://github.com/ribeiro-computer-vision/cvenv@v0.1.3"

import cvenv
print("cvenv", cvenv.__version__)
cvenv.PlatformManager().detect_platform()

## 2. Check what the wheel will be built against
The wheel is tied to these three values — a wheel is only reusable on a runtime with
the **same** python (cp), torch, and CUDA. Note them so you can name/track the wheel.

In [ ]:
import sys, torch
print(f"python : cp3{sys.version_info.minor}")
print(f"torch  : {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")
print(f"GPU    : {torch.cuda.is_available()}")

## 3. Mount Google Drive
So the wheel persists after the runtime resets. On Colab, `cvenv` saves wheels to
`/content/drive/MyDrive/cvenv_wheels/` by default.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Build the wheel
`build_wheel(...)` compiles from source (`FORCE_CUDA=1`) and saves the `.whl` **without
installing it** — it just produces the artifact. Use either the Python API *or* the
command line below; you don't need both.

### 4a. Python API

In [ ]:
WHEEL_DIR = "/content/drive/MyDrive/cvenv_wheels"   # default location; change if you like

whl = cvenv.get_component("pytorch3d").build_wheel(out_dir=WHEEL_DIR)
print("built:", whl)

### 4b. …or from the command line
Same thing as a shell command — handy as a one-off on any GPU box, not just in a
notebook. `--ref` picks the PyTorch3D branch/tag (default `stable`).

In [ ]:
!cvenv build-wheel pytorch3d --wheel-out-dir /content/drive/MyDrive/cvenv_wheels

## 5. Install and verify the wheel you built
Picks the newest wheel in the folder, so this works whether you built via the Python
API or the CLI above. `verify()` tests `import pytorch3d._C` — the real check that the
compiled extension matches this runtime's torch/CUDA.

In [ ]:
import glob, os
WHEEL = max(glob.glob("/content/drive/MyDrive/cvenv_wheels/pytorch3d-*.whl"),
            key=os.path.getmtime)
print("using:", WHEEL)

cvenv.get_component("pytorch3d").install(wheel_url=WHEEL)
cvenv.get_component("pytorch3d").verify()   # want: ✅ pytorch3d … (_C OK)

## 6. Reuse it later (or share it)
In any future session on a runtime with the **same** torch/CUDA/python, skip the build
entirely — point `wheel_url` at the saved file:

```python
import cvenv
cvenv.get_component("pytorch3d").install(
    wheel_url="/content/drive/MyDrive/cvenv_wheels/pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl")
```

The `.whl` in your Drive is a normal file — download it and give it to students, or
commit it somewhere.

**When Colab bumps torch/CUDA**, that wheel will fail `import pytorch3d._C` again — just
re-run this notebook to rebuild. Tip: keep one wheel per stack by naming the folder
after the build, e.g. `build_wheel(out_dir="/content/drive/MyDrive/cvenv_wheels/torch2.11_cu128_cp312")`.